# 00 — Data Infrastructure

**Strategy reference:** §17 (Data Contracts), §19 (Implementation Guidance).

Before we can study Tide, Wave, or Ripple, we need to understand
*where* the data lives and *how* to pull it.  The platform has two
stores:

* **OHLCV (Parquet, local or S3)** — bar data for every layer that
  works on L1.  Schema:
  `timestamp(ms) · open · high · low · close · volume [· spread]`.
  Files live under `{root}/ohlcv/{exchange}/{symbol}/{timeframe}.parquet`.
* **Ticks (HDF5)** — trades and per-side depth needed by Ripple.
  File `data/{exchange}_ticks.h5` contains three datasets per symbol:
  `trades`, `depth_snapshots`, and `depth_updates`.

Backend selection is environment-driven (`DATA_STORE=local_parquet|s3`).
Notebook helpers in `notebooks.utils` wrap both stores.

In [ ]:
# ── Data-source configuration ─────────────────────────────────────────
# OHLCV (Parquet) — S3 or local, controlled by DATA_STORE env var:
#   Local (default):  reads <project_root>/data/ohlcv/...
#   S3:               uncomment the two lines below
# import os
# os.environ["DATA_STORE"] = "s3"
# os.environ["S3_BUCKET"]  = "trading-data-centheos"
#
# Tick data (HDF5) — always stored locally; pull from S3 on demand:
#   load_ticks(...)              → use local cache (fast, no network)
#   load_ticks(..., refresh=True) → sync from S3 then read (ETag-gated)
#   Requires: AWS_PROFILE=trading (or AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY)
import os
os.environ["AWS_PROFILE"] = "trading"
os.environ["S3_BUCKET"]   = "trading-data-centheos"
# ─────────────────────────────────────────────────────────────────────

import sys, importlib
from pathlib import Path

_here = Path.cwd().resolve()
for _cand in [_here, *_here.parents]:
    if (_cand / "schemas.py").exists():
        _root = _cand; break
else:
    raise RuntimeError("Could not locate project root (no schemas.py found)")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import notebooks.utils as _utils_mod
importlib.reload(_utils_mod)   # always pick up on-disk changes without restarting the kernel

from notebooks.utils import (
    load_ohlcv, list_ohlcv, load_ticks, latest_book,
    plot_ohlcv, plot_equity_curve, configure_pandas, env_summary,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

configure_pandas()
%matplotlib inline

## 1. Environment
Inspect the env vars that drive `ohlcv_store.get_ohlcv_store()`.

In [ ]:
env_summary()

## 2. OHLCV inventory
`list_ohlcv` enumerates symbols available in **both** the Parquet
store *and* the legacy HDF5 file, so we always see what's reachable.

In [ ]:
exchange = 'binance'
inv = list_ohlcv(exchange)
print('Parquet symbols:', inv.get('parquet'))
print('HDF5  symbols:', inv.get('hdf5'))


## 3. Load OHLCV
`load_ohlcv` tries Parquet first (local or S3 depending on
`DATA_STORE`), then falls back to HDF5.  The returned DataFrame is
indexed by a UTC `DatetimeIndex`.

In [ ]:
symbol, timeframe = 'BTCUSDT', '1m'
ohlcv = load_ohlcv(exchange, symbol, timeframe)
print(f'rows={len(ohlcv):,}  cols={list(ohlcv.columns)}')
print(f'range: {ohlcv.index.min()}  →  {ohlcv.index.max()}')
ohlcv.head()

## 4. Data-quality scan
Gaps in bar data break every rolling-window feature downstream.
We sample a recent slice and look at the gap distribution.

In [ ]:
slice_ = ohlcv.tail(20_000)
gaps = slice_.index.to_series().diff().dt.total_seconds().dropna()
expected = 60.0
missing_bars = (gaps[gaps > expected * 1.5]).count()
print(f'window         : {slice_.index.min()} → {slice_.index.max()}')
print(f'expected step  : {expected}s')
print(f'median step    : {gaps.median():.1f}s')
print(f'gap >1.5× step : {int(missing_bars)} bars')
gaps.describe()

## 5. Candlestick + volume visual

In [ ]:
fig = plot_ohlcv(slice_.tail(1_000), title=f'{symbol} {timeframe} — last 1k bars')
plt.show()

## 6. Tick data
`load_ticks` opens `data/binance_ticks.h5` (or whatever path you
pass) and returns three DataFrames.  Depth rows use side `0=bid`,
`1=ask` — the same convention as the C++ `TickStore` writer.

In [ ]:
ticks = # refresh=False (default) — use local cache, no network access.
# refresh=True            — ETag-check S3 and download only if collector
#                           has uploaded new data since last refresh.
load_ticks(symbol, max_trades=5_000,
                   max_depth_snapshots=20_000,
                   max_depth_updates=20_000,
                   refresh=False)
for k, df in ticks.items():
    print(f'{k:18s} rows={len(df):,}')
ticks['trades'].head()

## 7. Reconstruct the most recent book
`latest_book` rebuilds the latest depth snapshot from the stored
rows.  This is the input the Ripple wall detector consumes.

In [ ]:
bids, asks, ts_ms = latest_book(ticks['depth_snapshots'], max_levels=10)
print('snapshot timestamp (ms):', ts_ms)
display(pd.concat({'bids': bids, 'asks': asks}, axis=1).head())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(bids['price'], bids['quantity'], height=1.0,
        color='#1f9d55', alpha=0.8, label='bids')
ax.barh(asks['price'], -asks['quantity'], height=1.0,
        color='#cc1f1a', alpha=0.8, label='asks')
ax.set_title(f'{symbol} order book — most recent snapshot')
ax.set_xlabel('quantity (negative = asks)')
ax.set_ylabel('price')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 8. Tick density (heatmap of trades / minute)

In [ ]:
trades = ticks['trades'].copy()
trades['ts'] = pd.to_datetime(trades['timestamp'], unit='ms')
per_min = trades.set_index('ts').resample('1min').size()
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(per_min.index, per_min.values, width=1/1440, color='#1f77b4')
ax.set_title('Trades per minute (recent tick window)')
ax.set_ylabel('count'); ax.grid(alpha=0.3)
plt.show()

## 9. Switching to S3
To pull OHLCV / ticks from S3 instead of local disk, set the env
vars *before* importing `notebooks.utils` (or restart the kernel):

```bash
export DATA_STORE=s3
export S3_BUCKET=my-trading-data-bucket
```

All notebooks downstream use the same `load_ohlcv` / `load_ticks`
calls, so the swap is transparent.
